In [1]:
import time
import requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
from datetime import datetime, timezone
from typing import List, Tuple
import json
import re
import pytz

# --------------------
# Configuration
# --------------------

HEADERS = {
    "User-Agent": "MyCustomCrawler/1.0 (+https://example.com/bot-info)"
}

CRAWL_DELAY = 1.5  # seconds
MAX_URLS = 200

NS = {"ns": "http://www.sitemaps.org/schemas/sitemap/0.9"}
PUBLISHER = "The Daily Caller"
BIAS = "right"

# --------------------
# Sitemap crawling
# --------------------
def normalize_text(text: str) -> str:
    if not text:
        return text

    replacements = {
        "“": '"',
        "”": '"',
        "‘": "'",
        "’": "'",
        "‚": "'",
        "‛": "'",
        "–": "-",
        "—": "-",
        "―": "-",
        "…": "...",
        "\u00a0": " ",   # non-breaking space
        "\u200b": "",    # zero-width space
    }

    for bad, good in replacements.items():
        text = text.replace(bad, good)

    return text
    
def fetch_xml(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    return resp.text


def parse_lastmod(value: str) -> datetime:
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except Exception:
        return datetime.min


def crawl_sitemap(sitemap_url, seen=None) -> List[Tuple[str, datetime]]:
    if seen is None:
        seen = set()

    if sitemap_url in seen:
        return []

    seen.add(sitemap_url)
    print(f"Fetching sitemap: {sitemap_url}")

    xml_text = fetch_xml(sitemap_url)
    root = ET.fromstring(xml_text)

    results = []

    # Sitemap index
    sitemap_tags = root.findall("ns:sitemap", NS)
    if sitemap_tags:
        for sitemap in sitemap_tags:
            loc = sitemap.find("ns:loc", NS)
            if loc is not None:
                time.sleep(CRAWL_DELAY)
                results.extend(crawl_sitemap(loc.text.strip(), seen))
        return results

    # Regular sitemap
    for url in root.findall("ns:url", NS):
        loc = url.find("ns:loc", NS)
        lastmod = url.find("ns:lastmod", NS)

        if loc is None:
            continue

        lastmod_dt = (
            parse_lastmod(lastmod.text.strip())
            if lastmod is not None and lastmod.text
            else datetime.min
        )

        results.append((loc.text.strip(), lastmod_dt))

    return results

def is_paywalled(soup) -> bool:
    # Hard stop: active paywall container
    if soup.find(id="paywallFluent"):
        return True

    paywall = soup.find("div", id="paywall")
    if not paywall:
        return False

    # Visible paywall via inline style
    style = paywall.get("style", "")
    if "display: inherit" in style or "display: block" in style:
        return True

    # Paywall card (older / fallback)
    if paywall.find("fluent-card", class_="old-paywall"):
        return True

    # Textual signals (belt & suspenders)
    text = paywall.get_text(" ", strip=True).lower()
    PAYWALL_PHRASES = (
        "premium article",
        "subscribe",
        "continue reading",
        "already have an account",
    )

    return any(phrase in text for phrase in PAYWALL_PHRASES)

# --------------------
# Article extraction
# --------------------

def fetch_html(url):
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    return resp.text
    
def normalize_published(date_str: str) -> str | None:
    """
    Converts 'January 14, 2026 4:47 PM ET' -> ISO-8601 UTC
    """
    if not date_str:
        return None

    try:
        # Remove trailing timezone label (ET)
        cleaned = re.sub(r"\s+[A-Z]{2}$", "", date_str)

        dt = datetime.strptime(cleaned, "%B %d, %Y %I:%M %p")

        eastern = pytz.timezone("US/Eastern")
        dt = eastern.localize(dt)

        return dt.astimezone(pytz.UTC).isoformat()
    except Exception:
        return None


def extract_article(url):
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    # Remove Twitter embeds completely
    for tweet in soup.find_all("blockquote", class_="twitter-tweet"):
        tweet.decompose()

    # ---- Paywall check ----
    if is_paywalled(soup):
        print(f"Skipping paywalled article: {url}")
        return None

    crawl_timestamp = datetime.now(timezone.utc).isoformat()

    # Headline
    headline_tag = soup.find(
        "h1",
        class_="mb-6 xl:mb-10 leading-tight font-oswald font-medium text-3xl md:text-5xl"
    )
    headline = normalize_text(headline_tag.get_text(strip=True)) if headline_tag else None

    # Published date & time
    published_raw = None
    published_iso = None
    
    time_tag = soup.find("time")
    if time_tag:
        spans = time_tag.find_all("span")
        if len(spans) >= 2:
            published_raw = f"{spans[0].get_text(strip=True)} {spans[1].get_text(strip=True)}"
            published_iso = normalize_published(published_raw)


    # Main body
    body_div = soup.find("div", id="articleContent")
    article_id = None
    body_text = ""

    if body_div:
        article_id = body_div.get("dc_data")

        paragraphs = body_div.find_all("p")
        cleaned = []

        for p in paragraphs:


            text = normalize_text(p.get_text(" ", strip=True))

            if not text:
                continue
        
            lowered = text.lower()
            
            # Skip RELATED content anywhere in the paragraph
            if "related:" in lowered or "(related:" in lowered:
                continue
                
            if "twitter.com" in lowered or "pic.twitter.com" in lowered:
                continue
                
            # Skip boilerplate
            if "daily caller news foundation" in lowered:
                continue
        
            cleaned.append(text)

        body_text = "\n\n".join(cleaned)

    word_count = len(body_text.split())

    # Skip very short articles
    if word_count < 120:
        print(f"Skipping short article ({word_count} words): {url}")
        return None

    return {
        "id": article_id,
        "url": url,
        "headline": headline,
        "published": published_iso,
        "published_raw": published_raw,
        "body": body_text,
        "word_count": word_count,
        "label": "right",
        "crawl_timestamp": crawl_timestamp,
        "outlet": PUBLISHER,
    }

# --------------------
# Main execution
# --------------------

if __name__ == "__main__":
    START_SITEMAPS = [
        "https://dailycaller.com/sitemap/4416",
        "https://dailycaller.com/sitemap/4415",
        "https://dailycaller.com/sitemap/4414",
        "https://dailycaller.com/sitemap/4413",
    ]
    OUTPUT_FILE = "dailycaller_200_articles.json"

    # Step 1: crawl multiple sitemaps
    all_entries = []

    for sitemap_url in START_SITEMAPS:
        all_entries.extend(crawl_sitemap(sitemap_url))

    # Step 2: sort by freshness
    all_entries.sort(key=lambda x: x[1], reverse=True)
    freshest_urls = [u for u, _ in all_entries]

    # Step 3: deduplicate URLs while preserving order (KEEP THIS)
    seen_urls = set()
    deduped_urls = []

    for url in freshest_urls:
        if url not in seen_urls:
            seen_urls.add(url)
            deduped_urls.append(url)

    freshest_urls = deduped_urls

    print(f"\nCrawling up to {MAX_URLS} valid articles\n")

    # Step 4: crawl articles until 100 valid ones are collected
    articles = []
    attempts = 0
    MAX_ATTEMPTS = len(freshest_urls)  # safety cap

    for url in freshest_urls:
        if len(articles) >= MAX_URLS or attempts >= MAX_ATTEMPTS:
            break

        attempts += 1
        print(f"[{attempts}] {url}")

        try:
            article = extract_article(url)
            if article:
                articles.append(article)
                print(f"✓ Collected {len(articles)}/{MAX_URLS}")
        except Exception as e:
            print(f"Failed: {e}")

        time.sleep(CRAWL_DELAY)

    if len(articles) < MAX_URLS:
        print(f"⚠️ Only collected {len(articles)} valid articles")

    # Step 5: dump to JSON
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        json.dump(
            articles,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"\nSaved {len(articles)} articles to {OUTPUT_FILE}\n")

Fetching sitemap: https://dailycaller.com/sitemap/4416
Fetching sitemap: https://dailycaller.com/sitemap/4415
Fetching sitemap: https://dailycaller.com/sitemap/4414
Fetching sitemap: https://dailycaller.com/sitemap/4413

Crawling up to 200 valid articles

[1] https://dailycaller.com/2026/03/17/nascar-daniel-dye-suspended-indycar-david-malukas-livestream/
✓ Collected 1/200
[2] https://dailycaller.com/2026/03/17/kat-abughazaleh-daniel-biss-illinois-house-primary/
✓ Collected 2/200
[3] https://dailycaller.com/2026/03/17/jb-pritzker-juliana-stratton-democratic-senate-primary-illinois/
✓ Collected 3/200
[4] https://dailycaller.com/2026/03/17/jesse-jackson-jr-donna-miller-illinois-house-primary/
✓ Collected 4/200
[5] https://dailycaller.com/2026/03/17/john-thune-gop-51-votes-talking-filibuster-save-act/
✓ Collected 5/200
[6] https://dailycaller.com/2026/03/17/opinion-three-takeaways-from-dhs-shutdown-jd-foster/
✓ Collected 6/200
[7] https://dailycaller.com/2026/03/17/indianapolis-colts-notre